# Full hand reconstruction from global quaternions

This notebook reconstructs a **full hand skeleton** from world-frame IMU quaternions stored in your glove CSV logs.  
It supports five fingers, extracts quaternions from a selected CSV row, computes joint locations with forward kinematics, and plots the hand in 3D.

Your attached CSV contains quaternion columns for left and right hand segments such as `leftindexproxquatw`, `leftindexproxquatx`, `leftindexproxquaty`, and `leftindexproxquatz`, along with analogous `mid` and wrist/palm columns. [file:41]

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from mpl_toolkits.mplot3d import Axes3D

In [8]:
def normalize_quaternion(q):
    q = np.asarray(q, dtype=float)
    n = np.linalg.norm(q)
    if n == 0:
        raise ValueError('Zero quaternion cannot be normalized')
    return q / n


def quat_to_rotmat(q, order='wxyz'):
    q = normalize_quaternion(q)
    if order == 'wxyz':
        w, x, y, z = q
    elif order == 'xyzw':
        x, y, z, w = q
    else:
        raise ValueError("order must be 'wxyz' or 'xyzw'")

    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),     1 - 2*(x*x + z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x*x + y*y)]
    ])


def relative_rotation(R_parent, R_child):
    return R_parent.T @ R_child

In [9]:
def segment_endpoint(origin, R_segment, length, axis_local=np.array([1.0, 0.0, 0.0])):
    axis_local = np.asarray(axis_local, dtype=float)
    axis_local = axis_local / np.linalg.norm(axis_local)
    return origin + R_segment @ (length * axis_local)


def fk_finger_global(base_point, rotations, lengths, axis_local=np.array([1.0, 0.0, 0.0])):
    points = {'base': np.asarray(base_point, dtype=float)}
    current = points['base']
    for seg_name, seg_length in lengths.items():
        R = rotations[seg_name]
        current = segment_endpoint(current, R, seg_length, axis_local)
        points[seg_name] = current
    return points

In [10]:
HAND_MODEL = {
    'thumb':  {'base_offset': np.array([0.020,  0.035, -0.010]), 'lengths': {'prox': 0.035, 'mid': 0.025, 'dist': 0.020}},
    'index':  {'base_offset': np.array([0.045,  0.020,  0.000]), 'lengths': {'prox': 0.045, 'mid': 0.025, 'dist': 0.018}},
    'middle': {'base_offset': np.array([0.050,  0.000,  0.000]), 'lengths': {'prox': 0.050, 'mid': 0.030, 'dist': 0.020}},
    'ring':   {'base_offset': np.array([0.047, -0.018,  0.000]), 'lengths': {'prox': 0.047, 'mid': 0.028, 'dist': 0.018}},
    'pinky':  {'base_offset': np.array([0.042, -0.035,  0.000]), 'lengths': {'prox': 0.038, 'mid': 0.020, 'dist': 0.016}},
}

In [11]:
def reconstruct_hand_from_global_quats(frame_quats, hand_model, quat_order='wxyz', axis_local=np.array([1.0, 0.0, 0.0]), distal_mode='inherit_mid'):
    R_palm = quat_to_rotmat(frame_quats['palm'], order=quat_order)
    palm_origin = np.zeros(3)
    hand_points = {'palm_origin': palm_origin, 'fingers': {}, 'relative_rotations': {}, 'palm_frame': R_palm}

    for finger_name, cfg in hand_model.items():
        base_point = palm_origin + R_palm @ cfg['base_offset']
        finger_quats = frame_quats[finger_name]

        rotations = {
            'prox': quat_to_rotmat(finger_quats['prox'], order=quat_order),
            'mid': quat_to_rotmat(finger_quats['mid'], order=quat_order)
        }
        if distal_mode == 'inherit_mid':
            rotations['dist'] = rotations['mid']
        else:
            rotations['dist'] = quat_to_rotmat(finger_quats['dist'], order=quat_order)

        points = fk_finger_global(base_point, rotations, cfg['lengths'], axis_local=axis_local)
        hand_points['fingers'][finger_name] = points
        hand_points['relative_rotations'][finger_name] = {
            'prox_to_mid': relative_rotation(rotations['prox'], rotations['mid']),
            'mid_to_dist': relative_rotation(rotations['mid'], rotations['dist'])
        }

    return hand_points

## Load a CSV and choose the row

This section lets you select a CSV file path and the row index to reconstruct. Your attached file has columns for palm/wrist quaternions and per-finger `prox` and `mid` quaternions for both left and right hands. [file:41]

In [12]:
CSV_PATH = '/home/jestin/ThesisRepo/ML/TwoHand_L_Fist_R_Fist_filtered_butterworth_lp/glove_data_L_Fist_R_Fist_5s_3_2026-04-19_19-31-18_bw_lp_5.0hz.csv'
HAND_SIDE = 'left'   # 'left' or 'right'
ROW_INDEX = 0        # choose any valid row index from the CSV
QUAT_ORDER = 'wxyz'

csv_path = Path(CSV_PATH)
if not csv_path.exists():
    raise FileNotFoundError(f'CSV not found: {csv_path.resolve()}')

df = pd.read_csv(csv_path)
print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Selected row:', ROW_INDEX)

Rows: 116
Columns: 295
Selected row: 0


In [13]:
def qcols(prefix):
    return [f'{prefix}quatw', f'{prefix}quatx', f'{prefix}quaty', f'{prefix}quatz']


def first_available_quaternion(row, prefixes, required=True):
    for prefix in prefixes:
        cols = qcols(prefix)
        if all(c in row.index for c in cols):
            return [row[c] for c in cols], prefix
    if required:
        raise KeyError(f'No quaternion columns found for any of: {prefixes}')
    return None, None


def row_to_frame_quats(df, row_index, hand_side='left'):
    row = df.iloc[row_index]
    side = hand_side.lower()
    if side not in {'left', 'right'}:
        raise ValueError("hand_side must be 'left' or 'right'")

    palm_q, palm_source = first_available_quaternion(
        row,
        [f'{side}wrist', f'{side}palmprox', f'{side}palmmid'],
        required=True
    )

    frame = {'palm': palm_q, '_sources': {'palm': palm_source}}

    for finger in ['thumb', 'index', 'middle', 'ring', 'pinky']:
        prox_q, prox_source = first_available_quaternion(row, [f'{side}{finger}prox'], required=False)
        mid_q, mid_source = first_available_quaternion(row, [f'{side}{finger}mid'], required=False)

        if prox_q is None and mid_q is None:
            continue

        if prox_q is None:
            prox_q = mid_q
            prox_source = mid_source + ' (fallback used for prox)'
        if mid_q is None:
            mid_q = prox_q
            mid_source = prox_source + ' (fallback used for mid)'

        frame[finger] = {'prox': prox_q, 'mid': mid_q}
        frame['_sources'][finger] = {'prox': prox_source, 'mid': mid_source}

    return frame


In [15]:
frame_quats = row_to_frame_quats(df, ROW_INDEX, hand_side=HAND_SIDE)
frame_quats['index']

KeyError: "No quaternion columns found for any of: ['leftwrist', 'leftpalmprox', 'leftpalmmid']"

In [ ]:
def plot_hand(hand, hand_model, title='Reconstructed hand pose'):
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')

    palm = hand['palm_origin']
    ax.scatter(*palm, s=60, c='k')
    ax.text(*palm, 'Palm')

    colors = {
        'thumb': 'tab:red',
        'index': 'tab:blue',
        'middle': 'tab:green',
        'ring': 'tab:orange',
        'pinky': 'tab:purple'
    }

    for finger_name, pts in hand['fingers'].items():
        chain_names = ['base'] + list(hand_model[finger_name]['lengths'].keys())
        chain = np.vstack([pts[name] for name in chain_names])
        ax.plot(chain[:, 0], chain[:, 1], chain[:, 2], '-o', linewidth=2.5, color=colors[finger_name], label=finger_name)
        ax.text(*pts['dist'], finger_name)

    palm_edge = np.vstack([hand['fingers'][name]['base'] for name in ['thumb', 'index', 'middle', 'ring', 'pinky']])
    ax.plot(palm_edge[:, 0], palm_edge[:, 1], palm_edge[:, 2], '--', color='gray', alpha=0.7)

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.legend()
    ax.set_box_aspect([1, 1, 0.7])
    plt.show()

In [ ]:
hand = reconstruct_hand_from_global_quats(
    frame_quats,
    HAND_MODEL,
    quat_order=QUAT_ORDER,
    distal_mode='inherit_mid'
)

plot_hand(hand, HAND_MODEL, title=f'{HAND_SIDE.capitalize()} hand reconstruction, row {ROW_INDEX}')

## Helpful utilities

You can inspect available rows, timestamps, and quickly switch frames by changing `ROW_INDEX`. The CSV also includes fields like `runindex` and request timestamps that can help you pick a specific sample. [file:41]

In [ ]:
# Preview a few useful columns if they exist
preview_cols = [c for c in ['runindex', 'requestid', 'requestts'] if c in df.columns]
if preview_cols:
    display(df[preview_cols].head(10))
else:
    print('No preview metadata columns found.')

In [ ]:
# Optional interactive row picker for Jupyter
try:
    import ipywidgets as widgets
    from IPython.display import display

    row_slider = widgets.IntSlider(value=ROW_INDEX, min=0, max=len(df)-1, step=1, description='Row')
    hand_dropdown = widgets.Dropdown(options=['left', 'right'], value=HAND_SIDE, description='Hand')

    def show_row(row_idx, hand_side):
        frame_quats = row_to_frame_quats(df, row_idx, hand_side=hand_side)
        hand = reconstruct_hand_from_global_quats(frame_quats, HAND_MODEL, quat_order=QUAT_ORDER, distal_mode='inherit_mid')
        plot_hand(hand, HAND_MODEL, title=f'{hand_side.capitalize()} hand reconstruction, row {row_idx}')

    out = widgets.interactive_output(show_row, {'row_idx': row_slider, 'hand_side': hand_dropdown})
    display(widgets.VBox([hand_dropdown, row_slider]), out)
except Exception as e:
    print('ipywidgets not available or not enabled:', e)

## Notes

This CSV may not always include every expected quaternion prefix, so the notebook now searches for the first available palm/root quaternion in this order: `wrist`, then `palmprox`, then `palmmid`. It also skips missing finger segments gracefully and can fall back from `mid` to `prox` or from `prox` to `mid` when only one of them exists in the selected row structure. [file:41]

